# Notebook 3 - Silver to Gold

# Notebook 3 - Silver -> Gold (Modelagem Dimensional + GenAI)

**Projeto:** CineData Analytics | Rocket Lab 2026 | **Camada:** Gold
**Stack:** Databricks + PySpark + Delta + Databricks Workflows (job.yaml)

---
## Objetivo
Construir a camada **Gold**: os **Data Marts** prontos para BI (modelagem **Star Schema**) e a
**tabela de contexto para o assistente de IA (RAG / Vector Index)**. Também respondemos às
**6 perguntas de negócio** do time de Analytics usando **display()**.

## Entregas
1. `gold.tb_dimensoes` (dims) + `gold.fact_movies_performance` (fato) + **bridge tables**.
2. `gold_genai_movies_context` — documento em linguagem natural para vetorização (RAG).
3. Visualizações que respondem às 6 perguntas de negócio.

---
## Modelagem dimensional (Star Schema)

### Dimensões
| Tabela | Colunas (PK/atributo) |
|---|---|
| `gold.dim_movies`     | `sk_movie_id` BIGINT PK · `id_filme` · `titulo` · `ano_lancamento` · `data_lancamento` · `duracao_minutos` · `idioma_original` · `status_filme` |
| `gold.dim_genres`     | `sk_genre_id` BIGINT PK · `nome_genero` |
| `gold.dim_people`     | `sk_person_id` BIGINT PK · `nome_pessoa` · `tipo_pessoa` (Ator/Diretor/Roteirista) |
| `gold.dim_companies`  | `sk_company_id` BIGINT PK · `nome_produtora` |
| `gold.dim_reviews`    | `sk_review_id` BIGINT PK · `sk_movie_id` FK · `qtd_avaliacoes` INT · `nota_media_usuarios` DOUBLE |

### Fato
| Tabela | Colunas |
|---|---|
| `gold.fact_movies_performance` | `sk_movie_id` PK/FK · `orcamento_usd` · `receita_usd` · `lucro_usd` · `orcamento_brl` · `receita_brl` · `lucro_brl` · `popularidade` · `nota_media_tmdb` · `qtd_votos_tmdb` · `nota_media_imdb` · `qtd_votos_imdb` |

### Bridge tables (devido ao relacionamento muitos-para-muitos)
| Tabela | Colunas |
|---|---|
| `gold.bridge_movie_genre`   | `sk_movie_id` FK · `sk_genre_id` FK |
| `gold.bridge_movie_person`  | `sk_movie_id` FK · `sk_person_id` FK |
| `gold.bridge_movie_company` | `sk_movie_id` FK · `sk_company_id` FK |

> Use **surrogate keys** (`monotonically_increasing_id()`, `row_number()` ou hash `sha2`) para não
> depender da chave natural de origem.


In [ ]:
# ============================================================
# 1) Datasets e funcoes de apoio
# ============================================================
gold_db     = "gold"
delta_format = "delta"
spark.sql(f"CREATE DATABASE IF NOT EXISTS {gold_db}")
print(f"Database `{gold_db}` pronto")


In [ ]:
# ============================================================
# 2) dim_movies (chave substituta sk_movie_id)
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_info = spark.table("silver.tb_info_filmes").select(
    "id_filme", "titulo", "ano_lancamento", "data_lancamento",
    "duracao_minutos", "idioma_original", "status_filme")

w = Window.orderBy("id_filme")
df_dim_movies = (df_info
   .withColumn("sk_movie_id", F.monotonically_increasing_id())
   .select("sk_movie_id", "id_filme", "titulo", "ano_lancamento",
           "data_lancamento", "duracao_minutos", "idioma_original", "status_filme"))

df_dim_movies.write.mode("overwrite").format(delta_format) \
     .saveAsTable(f"{gold_db}.dim_movies")
print(f"[GOLD] {gold_db}.dim_movies: {df_dim_movies.count():,}")


---
### Bridge tables
O relacionamento **muitos-para-muitos** entre `dim_movies` e as dimensões periféricas (gêneros,
pessoas, empresas) é resolvido com **bridge tables**, evitando duplicar o grão da Fato.


In [ ]:
# ============================================================
# 3) dim_genres -> bridge_movie_genre
# ============================================================
df_ge = (spark.table("silver.tb_generos")
    .distinct()
    .withColumn("sk_genre_id", F.monotonically_increasing_id())
    .select("sk_genre_id", "id_filme", "nome_genero"))

df_bridge_ge = df_ge.select("id_filme", "sk_genre_id")

df_ge.select("sk_genre_id", "nome_genero").drop("id_filme") \
    .distinct().write.mode("overwrite").format(delta_format) \
    .saveAsTable(f"{gold_db}.dim_genres")
df_bridge_ge.write.mode("overwrite").format(delta_format) \
    .saveAsTable(f"{gold_db}.bridge_movie_genre")
print(f"[GOLD] dim_genres / bridge_movie_genre [: {df_ge.select('id_filme').distinct().count():,} filmes")


In [ ]:
# ============================================================
# 4) dim_people (Ator/Diretor/Roteirista) + bridge
# ============================================================
df_pe = (spark.table("silver.tb_pessoas_empresas")
    .select("id_filme", "nome", "tipo")
    .withColumn("sk_person_id", F.monotonically_increasing_id()))

df_pe.select("sk_person_id", "nome", "tipo").distinct() \
    .write.mode("overwrite").format(delta_format) \
    .saveAsTable(f"{gold_db}.dim_people")
w = Window.orderBy("nome")
df_bridge_pe = (df_pe.select("id_filme", "sk_person_id").distinct())
df_bridge_pe.write.mode("overwrite").format(delta_format) \
    .saveAsTable(f"{gold_db}.bridge_movie_person")
print(f"[GOLD] dim_people / bridge_movie_person [: {df_pe.count():,}")


---
---
## 5) Fato `gold.fact_movies_performance`
Consolida métricas financeiras + engajamento em **um registro único por filme**, sem duplicar grão.

| Coluna | Tipo |
|---|---|
| `sk_movie_id` | BIGINT PK/FK |
| `orcamento_usd`, `receita_usd`, `lucro_usd` | DECIMAL(18,2) |
| `orcamento_brl`, `receita_brl`, `lucro_brl` | DECIMAL(18,2) |
| `popularidade` | DOUBLE |
| `nota_media_tmdb`, `nota_media_imdb` | DOUBLE |
| `qtd_votos_tmdb`, `qtd_votos_imdb` | INT |


In [ ]:
# ============================================================
# 5) fact_movies_performance (sem duplicar o grao do filme)
# ============================================================
df_fin = (spark.table("silver.tb_financeiro_filmes")
            .withColumnRenamed("id_filme", "id_filme_fin"))
df_met = (spark.table("silver.tb_metricas_engajamento")
            .withColumnRenamed("id_filme", "id_filme_met"))

df_fato = (df_dim_movies
   .join(df_fin, df_dim_movies["id_filme"] == df_fin["id_filme_fin"], "full")
   .join(df_met, F.coalesce(df_dim_movies["id_filme"], df_fin["id_filme_fin"])
                 == df_met["id_filme_met"], "left")
   .select(
      F.col("sk_movie_id"),
      F.coalesce(df_dim_movies["id_filme"], df_fin["id_filme_fin"],
                 df_met["id_filme_met"]).alias("id_filme"),
      df_fin["orcamento_usd"], df_fin["receita_usd"], df_fin["lucro_usd"],
      df_fin["orcamento_brl"], df_fin["receita_brl"], df_fin["lucro_brl"],
      df_met["popularidade"], df_met["nota_media_tmdb"], df_met["qtd_votos_tmdb"],
      df_met["nota_media_imdb"], df_met["qtd_votos_imdb"]))
df_fato.write.mode("overwrite").format(delta_format) \
    .saveAsTable(f"{gold_db}.fact_movies_performance")
print(f"[GOLD] {gold_db}.fact_movies_performance: {df_fato.count():,}")


---
---
## 6) `gold_genai_movies_context` — contexto para RAG (Vector Search / LLM)
Tabela de contexto do assistente de IA. A coluna **`llm_context_document`** é um **texto corrido
(frase natural)**, não uma lista de campos separados por vírgula.

**Modelo (frase):**
> "O filme [TITLE], lançado no ano de [ANO], faturou [RECEITA] e teve um custo de [BUDGET].
> Estrelado por [ATORES PRINCIPAIS] e dirigido por [DIRETOR], o filme possui a seguinte sinopse: [OVERVIEW]."

**Origem dos dados:** dim_movies (title, ano, overview) + fact_movies_performance (receita,
orçamento) + bridge_movie_person/dim_people (atores, diretor) — filtrando por `tipo_pessoa`.
Concentre os **atores principais** (agregados) numa única string **evitando NULL na concatenação**
(decisão → `coalesce` com fallback) para o documento não sumir silenciosamente.


In [ ]:
# ============================================================
# 6) gold_genai_movies_context (documento corrido p/ RAG)
# ============================================================
from pyspark.sql.functions import concat_ws, concat, lit, when, col

df_doc = (df_dim_movies
  .select("sk_movie_id","id_filme","titulo","ano_lancamento","data_lancamento","overview")
  .withColumn("overview", when(col("overview").isNull(), lit("Sem sinopse disponivel"))
                          .otherwise(col("overview")))
  .join(df_fato.select("sk_movie_id","receita_usd","orcamento_usd"),
        "sk_movie_id", "left")
  .withColumn("receita_usd", when(col("receita_usd").isNull(), lit(0)).otherwise(col("receita_usd")))
  .withColumn("orcamento_usd",when(col("orcamento_usd").isNull(), lit(0)).otherwise(col("orcamento_usd")))
)
# atores principais (agrupados) + diretor
df_cast = (spark.table("silver.tb_pessoas_empresas")
   .groupBy("id_filme")
   .agg(F.collect_list(when(col("tipo")=="Ator", col("nome"))).alias("atores"),
        F.max(when(col("tipo")=="Diretor", col("nome"))).alias("diretor")))
df_doc = (df_doc
  .join(df_cast, df_doc["id_filme"] == df_cast["id_filme"], "left")
  .withColumn("atores_txt", concat_ws(", ", col("atores")))
  .withColumn("ativo_fallback", when(col("atores_txt").isNull() |
      (col("atores_txt")==""), lit("Nao divulgado")).otherwise(col("atores_txt")))
  .withColumn("diretor_txt", when(col("diretor").isNull(),
      lit("Nao divulgado")).otherwise(col("diretor")))
  .withColumn("llm_context_document",
    concat_ws(" ",
      concat(lit("O filme "), col("titulo"),
             lit(", lancado no ano de "), col("ano_lancamento"),
             lit(", faturou "), col("receita_usd"),
             lit(" e teve um custo de "), col("orcamento_usd"),
             lit(". Estrelado por "), col("ativo_fallback"),
             lit(" e dirigido por "), col("diretor_txt"),
             lit(", o filme possui a seguinte sinopse: "), col("overview"),
             lit(".")))
)
df_doc.select("movie_id","title","llm_context_document") \
   .write.mode("overwrite").format(delta_format) \
   .saveAsTable(f"{gold_db}.gold_genai_movies_context")
print(f"[GOLD] {gold_db}.gold_genai_movies_context: {df_doc.count():,}")


---
---
## 7) Perguntas de negócio (entrega de Analytics)
Respondemos cada pergunta com `display()` — **sem bibliotecas de visualização**.


In [ ]:
# ============================================================
# Q1) Receita total (R$) somada de todos os filmes
# ============================================================
q1 = spark.sql("""
  SELECT FORMAT_NUMBER(SUM(receita_brl), 2) AS receita_total_brl
  FROM gold.fact_movies_performance
""")
display(q1)


In [ ]:
# ============================================================
# Q2) 5 filmes com maior popularidade
# ============================================================
q2 = spark.sql("""
  SELECT titulo, ROUND(popularidade, 2) AS popularidade
  FROM gold.fact_movies_performance f
  JOIN gold.dim_movies d ON f.sk_movie_id = d.sk_movie_id
  ORDER BY popularidade DESC
  LIMIT 5
""")
display(q2)


In [ ]:
# ============================================================
# Q3) Quantos filmes cada gênero possui (decrescente)
# ============================================================
q3 = spark.sql("""
  SELECT g.nome_genero, COUNT(*) AS qtd_filmes
  FROM gold.bridge_movie_genre b
  JOIN gold.dim_genres g ON b.sk_genre_id = g.sk_genre_id
  GROUP BY g.nome_genero
  ORDER BY qtd_filmes DESC
""")
display(q3)


In [ ]:
# ============================================================
# Q4) 10 filmes de maior receita + posição (RANK)
# ============================================================
q4 = spark.sql("""
  SELECT
    titulo,
    ROUND(receita_usd, 2)  AS receita_usd,
    ROUND(receita_brl, 2)  AS receita_brl,
    RANK() OVER (ORDER BY receita_usd DESC) AS posicao_rank
  FROM gold.fact_movies_performance f
  JOIN gold.dim_movies d ON f.sk_movie_id = d.sk_movie_id
  ORDER BY posicao_rank
  LIMIT 10
""")
display(q4)


In [ ]:
# ============================================================
# Q5) Ator com mais participações nos últimos 2 anos
# ============================================================
q5 = spark.sql("""
  SELECT
    p.nome_pessoa,
    COUNT(DISTINCT b.sk_movie_id) AS qtd_participacoes
  FROM gold.bridge_movie_person b
  JOIN gold.dim_people  p ON b.sk_person_id = p.sk_person_id
  JOIN gold.dim_movies  d ON b.sk_movie_id  = d.sk_movie_id
  WHERE p.tipo_pessoa  = 'Ator'
    AND d.data_lancamento >= ADD_MONTHS(
        (SELECT MAX(data_lancamento) FROM gold.dim_movies), -24)
  GROUP BY p.nome_pessoa
  ORDER BY qtd_participacoes DESC
  LIMIT 1
""")
display(q5)


In [ ]:
# ============================================================
# Q6) Produtora de filmes com maior Lucro nos últimos 5 anos
# ============================================================
q6 = spark.sql("""
  SELECT
    c.nome_produtora,
    SUM(f.lucro_usd) AS lucro_total_usd
  FROM gold.fact_movies_performance f
  JOIN gold.bridge_movie_company b ON b.sk_movie_id = f.sk_movie_id
  JOIN gold.dim_companies      c ON c.sk_company_id = b.sk_company_id
  JOIN gold.dim_movies         d ON d.sk_movie_id   = f.sk_movie_id
  WHERE d.data_lancamento >= ADD_MONTHS(
        (SELECT MAX(data_lancamento) FROM gold.dim_movies), -60)
  GROUP BY c.nome_produtora
  ORDER BY lucro_total_usd DESC
  LIMIT 1
""")
display(q6)
print("[FIM] Camada Gold concluída.")
